# Удержание абонентов: стартовый ноутбук

Задача — помочь команде удержания выбрать 15% абонентов для контакта на дату снимка. Цель `churn_30d` относится к следующим 30 дням. Ноутбук содержит точки старта, но не готовое полное решение.

In [ ]:
from pathlib import Path
import sys
import pandas as pd

case_candidates = [Path.cwd(), Path.cwd() / 'examples' / 'synthetic-case', Path.cwd().parent]
CASE_ROOT = next(path for path in case_candidates if (path / 'generate_data.py').exists())
sys.path.insert(0, str(CASE_ROOT))

from src.data_preparation import FORBIDDEN_FEATURES, SAFE_FEATURES, load_data

DATA_PATH = CASE_ROOT / 'data' / 'subscriber_retention.csv'
data = load_data(DATA_PATH)
data.shape

In [ ]:
display(data.head())
display(data.dtypes.to_frame('dtype'))
print('Доля цели:', data['churn_30d'].mean())
print('Период:', data['snapshot_date'].min(), '—', data['snapshot_date'].max())

## Проверка доступности признаков

`leaked_churn_score` и `retention_offer_result_14d` недоступны на дату решения. Не удаляйте их молча: объясните тип утечки и зафиксируйте исключение в журнале решений.

In [ ]:
print('Безопасные признаки:', SAFE_FEATURES)
print('Запрещённые признаки:', FORBIDDEN_FEATURES)

# TODO: проверьте, нет ли других полей, недоступных в выбранном вами сценарии.

## EDA

TODO:

1. Опишите единицу наблюдения и временную структуру.
2. Найдите пропуски, точные дубликаты и выбросы.
3. Сравните долю оттока по времени и содержательным группам.
4. Сформулируйте минимум три проверяемые гипотезы.
5. Отделяйте наблюдение от предметной интерпретации.

In [ ]:
quality_summary = pd.DataFrame({
    'missing': data.isna().sum(),
    'missing_share': data.isna().mean(),
})
display(quality_summary.sort_values('missing_share', ascending=False).head(10))
print('Точные дубликаты:', data.duplicated().sum())

# TODO: добавьте содержательные проверки диапазонов и графики.

## Подготовка и разбиение

Основной протокол должен проверять будущие месяцы на модели, обученной на прошлом. Параметры заполнения и преобразований нельзя вычислять по контрольному периоду.

In [ ]:
# TODO: удалите точные дубликаты.
# TODO: выберите cutoff и получите непересекающиеся train/test по времени.
# TODO: обоснуйте обработку пропусков и выбросов только на train.
# TODO: соберите pipeline без FORBIDDEN_FEATURES.

## Моделирование и оценка

Сначала зафиксируйте наивный baseline. Затем сравните минимум две содержательно разные модели на одинаковом разбиении. Помимо ROC-AUC используйте Average Precision и метрики при ограничении контакта верхними 15% риска.

In [ ]:
experiment_log = pd.DataFrame(columns=[
    'run_id', 'data_version', 'cutoff_date', 'features', 'model',
    'parameters', 'roc_auc', 'average_precision',
    'precision_at_top_15pct', 'recall_at_top_15pct', 'comment'
])
experiment_log

## Итоговые материалы

В проекте должны остаться паспорт данных, журнал решений, воспроизводимый pipeline, журнал экспериментов, анализ ошибок, ограничения и рекомендация для руководителя команды удержания. Все числа в отчёте должны повторяться из кода.